# 18. 롱숏 샤프 비율 (Long-Short Sharpe Ratio)

17번에서 Rank IC가 "순서를 얼마나 잘 맞히나"였다면, 이번에는 **"그 시그널로 실제 포트폴리오를 만들면 위험 대비 돈을 얼마나 버나"**야. 순서는 이래.

1. 샤프 비율의 정의
2. 롱숏 포트폴리오 구성
3. 숫자 예시와 연율화
4. 샤프 비율의 추정 오차
5. Rank IC와 롱숏 수익의 연결
6. 거래비용과 회전율
7. 한국 시장의 공매도 현실
8. 샤프 비율 너머: 팩터 알파, 꼬리 위험
9. 레짐별 성과 비교
10. 함정과 코드

---

## 0단계: 샤프 비율이란

수익률만 보면 위험을 무시하게 돼. 연 20% 버는 전략이라도 매년 ±50%씩 흔들린다면 좋은 전략이 아니지. 샤프 비율은 **위험 한 단위당 초과수익**이야.

$$
SR = \frac{E[R_p - R_f]}{\sigma(R_p - R_f)}
$$

- $R_p$: 포트폴리오 수익률
- $R_f$: 무위험 수익률 (예: CD 금리, 통안채 금리)
- 분자: 무위험 자산보다 평균적으로 얼마나 더 벌었나
- 분모: 그 초과수익이 얼마나 흔들렸나

17번의 ICIR과 같은 구조야. **평균을 변동성으로 나눈 것**이지. 5번 모멘트 표준화 때처럼 단위를 없애서, 변동성이 다른 전략들을 같은 척도로 비교할 수 있게 해줘.

---

## 1단계: 롱숏 포트폴리오 구성

매 리밸런싱 시점(예: 매주 금요일 장 마감 후)마다 이렇게 해.

1. 모든 종목을 시그널 기준으로 정렬한다.
2. 10분위(decile)로 나눈다.
3. **예측 수익이 높은 분위를 사고(롱), 낮은 분위를 판다(숏).**

RSJ는 IC가 음수였으니, **RSJ 하위 10%를 롱, 상위 10%를 숏**하는 거야.

비중은 보통 이렇게 정해.

$$
\sum_{i \in \text{롱}} w_i = +1, \qquad \sum_{i \in \text{숏}} w_i = -1
$$

롱에 1억, 숏에 1억을 배분하는 거야. 순투자 금액이 0이라서 **달러 중립(dollar-neutral)** 또는 **제로 투자(zero-investment) 포트폴리오**라고 불러. 롱 안에서는 동일가중(각 종목 1/n)이나 시가총액가중을 써.

롱숏 수익률은 두 다리(leg)의 차이야.

$$
R_{LS,t+1} = R_{\text{롱},t+1} - R_{\text{숏},t+1}
$$

**롱숏에서는 왜 무위험 수익률을 안 빼나?** 이론적으로는 숏으로 판 돈으로 롱을 사니까 **자기 자금이 들지 않아.** 자기 돈을 안 쓴 전략은 무위험 자산에 넣어둘 기회비용도 없으니, $R_{LS}$ 자체가 이미 초과수익이야. 그래서 학술 연구의 관행은 이래.

$$
SR_{LS} = \frac{E[R_{LS}]}{\sigma(R_{LS})}
$$

(실제로는 증거금에 이자가 붙고 대차 비용이 들지만, 그건 6·7단계에서 다룰게.)

**롱숏의 핵심 장점:** 시장이 폭락해서 롱이 −10%, 숏도 −12%면 롱숏은 **+2%**야. 시장 방향(0번 개념에서 본 시계열 예측)을 맞힐 필요 없이, **상대적 순서만 맞히면** 돈을 벌어. 17번 0단계의 횡단면 예측을 돈으로 바꾸는 장치가 바로 이거야.

---

## 2단계: 숫자 예시와 연율화

5주간의 결과야(%).

| 주 | 롱 | 숏 | 롱숏 |
|---|---|---|---|
| 1 | +1.2 | +0.4 | +0.8 |
| 2 | −2.0 | −2.5 | +0.5 |
| 3 | +0.3 | +0.9 | −0.6 |
| 4 | +2.1 | +1.3 | +0.8 |
| 5 | −0.5 | −1.0 | +0.5 |

2주차를 봐. 롱도 숏도 다 떨어졌지만(시장 하락), 롱숏은 +0.5%야.

**주간 샤프 비율:**
- 평균: $(0.8 + 0.5 - 0.6 + 0.8 + 0.5)/5 = 0.40\%$
- 표준편차: 편차가 $0.4, 0.1, -1.0, 0.4, 0.1$이니 제곱합이 $0.16 + 0.01 + 1.00 + 0.16 + 0.01 = 1.34$이고, $\sqrt{1.34/4} = 0.579\%$
- 주간 SR = $0.40 / 0.579 = 0.69$

**연율화 (유도):** 주간 수익률이 서로 독립이라고 가정하면, 1년(52주) 동안

- 평균은 52배로 늘어나: $52\mu$
- 분산도 52배로 늘어나니, 표준편차는 $\sqrt{52}$배: $\sqrt{52}\,\sigma$

$$
SR_{\text{연}} = \frac{52\mu}{\sqrt{52}\,\sigma} = \sqrt{52}\cdot SR_{\text{주}}
$$

$$
SR_{\text{연}} = \sqrt{52} \times 0.69 \approx \mathbf{5.0}
$$

**연 5.0은 비현실적인 숫자야.** 5주짜리 예시라 우연히 높게 나온 거고(4단계 참고), 거래비용도 빠졌어(6단계 참고). 참고로 실제 주식 시장 전체의 연 샤프는 0.3~0.5 정도고, 거래비용 차감 후 연 1.0을 꾸준히 넘는 전략은 매우 드물어.

**√52 규칙의 한계:** 주간 수익률에 자기상관이 있으면 이 공식이 틀려. 양의 자기상관(이번 주 잘 되면 다음 주도 잘 됨)이 있으면 실제 연간 변동성이 √52배보다 커서 연율 샤프가 **과대평가**돼. 유동성이 낮은 종목이 많은 전략에서 흔한 문제야(Lo 2002).

---

## 3단계: 샤프 비율도 추정치다 — 표준오차

샤프 비율은 표본에서 계산한 **추정치**라 오차가 있어. 수익률이 독립이고 정규분포에 가까우면(Lo 2002), T개 기간으로 계산한 샤프의 표준오차는 대략

$$
\text{SE}(\widehat{SR}) \approx \sqrt{\frac{1 + \frac{1}{2}SR^2}{T}}
$$

(SR과 T는 같은 기간 단위로 맞춰야 해. 연 SR이면 T는 연수.)

| 진짜 연 SR | 기간 | 표준오차 | 95% 신뢰구간 |
|---|---|---|---|
| 1.0 | 3년 | 0.71 | −0.4 ~ 2.4 |
| 1.0 | 10년 | 0.39 | 0.2 ~ 1.8 |
| 0.5 | 10년 | 0.34 | −0.2 ~ 1.2 |

**10년 데이터로도 연 샤프 0.5 전략은 "0과 구별되지 않을" 수 있어.** 대략 $t \approx SR_{\text{연}} \times \sqrt{\text{연수}}$라서, t = 2를 넘으려면 10년 기준 연 샤프 0.63 이상이 필요해.

12번 5단계의 **선택 편향**이 여기서도 똑같이 적용돼. λ 후보 20개, 시그널 변형 10개, 분위 기준 3개를 시도해서 가장 좋은 샤프를 보고하면, 실력이 없어도 높은 샤프가 나와. 이걸 보정하는 방법으로 **Deflated Sharpe Ratio**(Bailey & López de Prado 2014)가 있어. **몇 번 시도했는지를 감안해서** 샤프 비율의 유의성을 낮춰 평가하는 방법이야. 이름만 기억해뒀다가 나중에 찾아봐.

---

## 4단계: Rank IC와 롱숏 수익의 연결 (유도)

17번과 18번을 잇는 공식이야. 매 시점 표준화한 시그널 $z_i$(평균 0, 표준편차 1)와 수익률 사이에 선형 관계가 있다고 하자. 회귀 기울기는 **상관 × (y의 표준편차 / x의 표준편차)**니까:

$$
E[r_i \mid z_i] = \bar{r} + IC \cdot \sigma_{CS} \cdot z_i
$$

$\sigma_{CS}$는 그 기간 종목 수익률의 **횡단면** 표준편차야(종목끼리 얼마나 수익률이 흩어졌나).

롱숏 수익의 기댓값은 두 분위의 평균 z-score 차이에 비례해.

$$
E[R_{LS}] \approx IC \cdot \sigma_{CS} \cdot (\bar{z}_{\text{롱}} - \bar{z}_{\text{숏}})
$$

시그널이 정규분포를 따르면 상위 10% 구간의 평균 z-score는 약 1.75야(표준정규분포에서 상위 10% 경계 1.28 위쪽 구간의 평균). 하위 10%는 −1.75지. 그래서 차이가 약 **3.5**야.

**숫자 예시:** Rank IC 0.03, 주간 횡단면 수익률 표준편차 5%라면

$$
E[R_{LS}] \approx 0.03 \times 5\% \times 3.5 \approx \mathbf{0.53\%/\text{주}}
$$

IC 0.03이 작아 보여도 주당 0.5% 넘는 스프레드로 이어질 수 있어. 연간으로 치면 25% 이상이야. 하지만 다음 단계의 거래비용이 이걸 거의 다 잡아먹을 수 있어.

이 공식이 주는 통찰이 하나 더 있어. **같은 IC라도 $\sigma_{CS}$가 큰 시기에 롱숏 수익이 커져.** 혼란 레짐에서는 종목 간 수익률 격차가 벌어지니, IC가 그대로여도 롱숏 수익은 커질 수 있어. 레짐별 결과를 해석할 때 IC 차이와 $\sigma_{CS}$ 차이를 구분해야 하는 이유야.

---

## 5단계: 거래비용 — 고빈도 시그널의 무덤

**회전율(turnover):** 리밸런싱마다 포트폴리오의 얼마나 바뀌나.

$$
TO_t = \sum_i \big|w_{i,t} - w_{i,t^-}\big|
$$

$w_{i,t^-}$는 리밸런싱 **직전** 비중이야(지난주 비중에서 가격 변동만 반영된 것). 매주 롱 종목의 절반이 바뀌면 롱 다리 회전율은 대략 50%(판 것 + 산 것을 합산하는 방식에 따라 100%로 세기도 하니, 정의를 명시해야 해).

**순수익 (거래비용 차감 후):**

$$
R_{LS,t}^{\text{net}} = R_{LS,t} - c \cdot TO_t
$$

c는 거래 1단위당 비용이야. 한국에서는 이런 것들이 포함돼.

- **증권거래세** (매도 시): 대략 0.15~0.2% 수준인데 연도별로 세율이 바뀌어 왔으니, 분석 기간별로 확인해서 적용해야 해.
- 위탁수수료
- 호가 스프레드 (매수호가와 매도호가의 차이의 절반)
- 시장충격 (내 주문이 가격을 밀어내는 비용, 소형주일수록 큼)

**숫자로 체감하기:** 왕복(사고 팔기) 비용 0.3%, 매주 각 다리의 절반을 교체한다면

$$
\text{주간 비용} \approx 2\text{개 다리} \times 50\% \times 0.3\% = 0.30\%
$$

4단계에서 계산한 주간 총수익 0.53% 중 **0.30%가 비용으로 사라져.** 연간 기준 약 15%야. 소형주 시장충격까지 넣으면 순수익이 0 이하가 될 수도 있어.

**손익분기 비용:** 총수익이 비용으로 딱 상쇄되는 거래비용을 계산해서 보고하는 것도 좋은 방법이야.

$$
c^* = \frac{E[R_{LS}]}{E[TO]}
$$

"이 전략은 왕복 비용이 0.5%보다 낮아야 수익이 난다" 같은 식으로 결과를 해석할 수 있어.

**핵심 함의:** 장중 데이터로 만든 시그널(RSJ, RSkew)은 17번에서 봤듯 IC가 빨리 감소해서 **자주 리밸런싱**해야 하고, 그래서 비용에 가장 취약해. 이런 시그널로 "총수익 기준 샤프 2.0"을 보고하면, 면접관이 가장 먼저 "비용 차감 후엔요?"라고 물을 거야.

**비용 줄이는 방법:** 리밸런싱 주기 늘리기, 시그널을 몇 주 평균으로 평활화하기, 기존 보유 종목이 경계 근처면 유지하기(buffer), 유동성 높은 종목으로 universe 제한하기.

---

## 6단계: 한국 시장의 공매도 현실

학술 논문의 롱숏은 "원하는 종목을 원하는 만큼 공매도할 수 있다"고 가정해. 한국에서는 이 가정이 특히 약해.

- **2023년 11월 ~ 2025년 3월:** 공매도 전면 금지. 이 기간 롱숏 수익은 **달성 불가능한 숫자**야.
- **대차 가능성:** 소형주, 코스닥 종목은 빌릴 주식 자체가 없는 경우가 많아.
- **대차 비용:** 빌릴 수 있어도 인기 공매도 종목은 수수료가 비싸.

그런데 IVOL, RSJ 같은 시그널에서 **숏 쪽 종목은 바로 이런 종목들**(소형, 고변동, 개인 선호)이야. 즉 수익의 상당 부분이 **실제로 공매도하기 가장 어려운 종목**에서 나올 가능성이 커. 5번의 Stambaugh-Yu-Yuan 논리와 정확히 같은 이야기지. 공매도가 어려우니 고평가가 유지되고, 그래서 숏 수익이 "존재하지만 가져갈 수 없는" 거야.

**그래서 반드시 이렇게 보고해야 해.**

**(1) 다리별 분해:** 롱숏 수익을 롱 다리 알파와 숏 다리 알파로 나눠봐. 수익이 대부분 숏 다리에서 나온다면 실현 가능성이 낮다는 뜻이야.

**(2) 실현 가능한 대안 전략:**

| 대안 | 구성 | 평가 지표 |
|---|---|---|
| 롱온리 | 시그널 상위 분위만 매수 | 벤치마크(KOSPI) 대비 초과수익의 정보비율 |
| 롱 + 선물 헤지 | 상위 분위 매수 + KOSPI200 선물 매도 | 헤지 후 샤프 |
| 대형주 롱숏 | KOSPI200 종목만으로 롱숏 | 샤프 (대차가 비교적 쉬운 종목) |

---

## 7단계: 샤프 비율 너머

**팩터 알파: 새로운 수익인가, 알려진 효과인가**

17번의 중립화와 같은 질문을 수익률 수준에서 해. 롱숏 수익률을 알려진 팩터들에 회귀해.

$$
R_{LS,t} = \alpha + \beta_{MKT}MKT_t + \beta_{SMB}SMB_t + \beta_{HML}HML_t + \beta_{REV}REV_t + \varepsilon_t
$$

5번에서 본 Fama-French 팩터에 단기반전 팩터(REV)까지 넣었어. **α가 유의하면** 기존 팩터로 설명되지 않는 새로운 수익이라는 뜻이야. RSJ 롱숏의 샤프가 높아도 α가 0이라면, 그건 반전 팩터를 비싸게 복제한 것에 불과해.

**꼬리 위험:** 샤프 비율은 평균과 표준편차만 봐서 **분포의 모양을 무시해.** 1번에서 배운 왜도를 떠올려봐.

- 평소에 조금씩 벌다가 가끔 크게 잃는 전략(음의 왜도)은 샤프가 **실제보다 좋아 보여.**
- 모멘텀 전략이 대표적이야. 평소엔 잘 되다가 시장 급반등기에 폭락(momentum crash)하지.

그래서 샤프와 함께 이것들을 보고해.

- **최대 낙폭(maximum drawdown):** 고점 대비 가장 크게 떨어진 폭
- **왜도, 첨도:** 1번과 모멘트 개념에서 배운 그것
- **소르티노 비율:** 분모를 전체 변동성 대신 **하방 변동성**으로 바꾼 샤프. 2번 semivariance와 같은 발상이야.

$$
\text{Sortino} = \frac{E[R]}{\sqrt{E[\min(R, 0)^2]}}
$$

**동일가중 vs 시가총액가중:** 동일가중은 소형주 비중이 커서 결과가 부풀려지기 쉬워. 5번에서 말했듯 **둘 다 보고**하고, 시가총액가중에서도 결과가 유지되는지 확인해.

---

## 8단계: 레짐별 성과 비교

17번 9단계와 같은 구조를 수익률에 적용하면 돼.

$$
R_{LS,t+1} = a + b\,D_t + e_{t+1}
$$

$D_t$는 16번의 **online** 레짐 더미야(t시점까지의 정보). $b$가 레짐 간 평균 수익 차이고, Newey-West 표준오차로 검정해.

레짐별 샤프도 따로 계산해서 보여줄 수 있어. 다만 **두 샤프 비율의 차이를 공식적으로 검정**하는 건 평균 차이 검정보다 까다로워. Ledoit & Wolf(2008)의 부트스트랩 방법 같은 게 필요하다는 정도만 알아둬. 실무에서는 평균 차이(b)를 주 검정으로 쓰고, 레짐별 샤프는 기술통계로 보여주는 경우가 많아.

4단계에서 말한 해석상의 주의도 기억해. 혼란 레짐에서 롱숏 수익이 크게 나왔다면, 그게 **IC가 커져서**인지 **횡단면 분산($\sigma_{CS}$)이 커져서**인지 구분해야 해. 17번의 레짐별 IC 결과와 같이 보면 구분할 수 있어.

---

## 9단계: 함정

**리밸런싱 시점과 수익률 시점.** 16번 6단계, 17번 10단계와 같아. 금요일 장 마감 후 계산한 시그널로는 **월요일 시가 이후** 수익률만 쓸 수 있어. 금요일 종가로 체결했다고 가정하면 look-ahead야. 더 보수적으로 가려면 **다음 거래일 종가 체결**을 가정해.

**상장폐지 수익률 누락.** 17번과 같아. 숏 다리에 있던 종목이 상폐되면 그게 숏의 가장 큰 수익인데, 데이터에서 빠지면 전략 성과가 **과소평가**돼. 반대로 롱 다리 종목이 상폐되면 손실이 빠져서 **과대평가**되고.

**거래 불가능한 가격.** 상한가에 묶인 종목은 살 수 없고, 하한가 종목은 팔 수 없어. 거래정지 종목도 마찬가지야. 이런 종목을 해당 기간에 포트폴리오에 편입하거나 청산했다고 가정하면 안 돼.

**샤프 비율 비교의 단위.** 주간, 월간, 연간 샤프를 섞어서 비교하는 실수가 흔해. 항상 **연율화된 값**으로 통일하고, 어떤 주기로 계산했는지 명시해.

---

## 계산 코드

```python
import numpy as np
import pandas as pd
import statsmodels.api as sm

def long_short_backtest(panel, n_q=10, sign=-1, cost_rt=0.003, weight='ew'):
    """
    panel: [date, ticker, signal, fwd_ret, mcap]
      - date: 리밸런싱 시점, fwd_ret: 다음 리밸런싱까지 수익률 (실행 지연 반영된 것)
    sign: IC 부호 (RSJ는 -1 → 시그널 낮은 쪽을 롱)
    cost_rt: 왕복 거래비용
    """
    panel = panel.dropna(subset=['signal', 'fwd_ret']).copy()
    panel['q'] = panel.groupby('date')['signal'].transform(
        lambda x: pd.qcut(x.rank(method='first'), n_q, labels=False))
    long_q, short_q = (0, n_q - 1) if sign < 0 else (n_q - 1, 0)

    def leg_weights(g, q):
        g = g[g['q'] == q]
        w = g['mcap'] if weight == 'vw' else pd.Series(1.0, index=g.index)
        return pd.Series((w / w.sum()).values, index=g['ticker'].values)

    rows, prev_w = [], None
    for d, g in panel.groupby('date'):
        wL, wS = leg_weights(g, long_q), leg_weights(g, short_q)
        w = wL.sub(wS, fill_value=0)                       # 롱 +, 숏 -
        r = g.set_index('ticker')['fwd_ret']
        gross = (w * r.reindex(w.index).fillna(0)).sum()
        # 회전율 (단순화: 직전 비중 대비 절대 변화의 합, 가격 드리프트 무시)
        to = w.abs().sum() if prev_w is None else \
             w.sub(prev_w, fill_value=0).abs().sum()
        rows.append({'date': d, 'gross': gross, 'turnover': to,
                     'net': gross - cost_rt / 2 * to})     # 편도 비용 × 거래량
        prev_w = w
    return pd.DataFrame(rows).set_index('date')

def perf_summary(r, periods_per_year=52):
    mu, sd = r.mean(), r.std()
    sr = mu / sd * np.sqrt(periods_per_year)
    yrs = len(r) / periods_per_year
    wealth = (1 + r).cumprod()
    return pd.Series({
        'ann_ret': mu * periods_per_year,
        'ann_vol': sd * np.sqrt(periods_per_year),
        'sharpe': sr,
        'sharpe_se': np.sqrt((1 + 0.5 * sr**2) / yrs),
        'sortino': mu / np.sqrt((np.minimum(r, 0)**2).mean()) * np.sqrt(periods_per_year),
        'max_dd': (wealth / wealth.cummax() - 1).min(),
        'skew': r.skew(),
        'avg_turnover': np.nan,
    })

bt = long_short_backtest(panel, sign=-1, cost_rt=0.003)
print(perf_summary(bt['gross']), perf_summary(bt['net']))
print('손익분기 왕복비용:', 2 * bt['gross'].mean() / bt['turnover'].mean())

# 팩터 알파
X = sm.add_constant(factors.loc[bt.index, ['MKT', 'SMB', 'HML', 'REV']])
alpha = sm.OLS(bt['net'], X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})

# 레짐별 평균 수익 차이 (D: online 레짐 더미, 리밸런싱 시점 기준)
Xr = sm.add_constant(regime_online.reindex(bt.index))
reg = sm.OLS(bt['net'], Xr, missing='drop').fit(cov_type='HAC', cov_kwds={'maxlags': 4})
```

회전율 계산은 단순화한 버전이야. 정확하게 하려면 직전 비중에 기간 수익률을 반영해 "드리프트된 비중"을 먼저 구하고 차이를 계산해야 해.

---

## 네 프로젝트의 보고 순서 제안

17번과 18번을 합쳐서, 시그널 하나에 대한 결과를 이 순서로 정리하면 설득력 있는 구조가 돼.

1. **Rank IC** 평균, ICIR, Newey-West t-값 (17번)
2. **IC decay** 예측 기간별 (17번)
3. **중립화 후 IC** (반전, 규모, 업종 통제) (17번)
4. **10분위 수익률**이 단조적으로 증가·감소하는가
5. **롱숏 샤프**: 총수익과 거래비용 차감 후, 동일가중과 시가총액가중 (18번)
6. **다리별 분해**와 **실현 가능한 대안 전략** (18번)
7. **팩터 알파** (18번)
8. **레짐별** IC 차이와 롱숏 수익 차이 (17, 18번)
9. **공매도 금지 기간** 별도 분석

---

**한 줄 요약:** 롱숏 샤프 비율은 시그널 상위 분위를 사고 하위 분위를 판 **제로 투자 포트폴리오의 평균 수익 ÷ 변동성**이야. 연율화는 √(연간 기간 수)를 곱하면 되지만, 추정 오차가 커서 10년 데이터로도 연 샤프 0.5는 0과 구별이 안 될 수 있어. 가장 중요한 건 **거래비용과 공매도 가능성**이야. 장중 시그널은 회전율이 높아 비용에 특히 취약하고, 한국에서는 숏 쪽 종목을 실제로 빌리기 어려워서, 총수익 기준 샤프만 보고하면 실현 불가능한 숫자가 될 가능성이 높아.

마지막 19번 **레짐 조건부 / 무조건부 모형**은 지금까지의 모든 내용을 연구 설계로 묶는 개념이야. 준비되면 말해줘.